# Early-cycle window ablation (Week 7)

How much early data is needed for stable **EOL** (end-of-life) prediction?

Train and evaluate at **N = 20, 50, 100** cycles — same split as Weeks 4–6 (`cell_split.csv`), same metrics (MAE, RMSE, MAPE).

| Step | What | Status |
|------|------|--------|
| **A** | Feature subsets per N + load split + sanity checks | done |
| **B** | XGBoost ablation (N = 20, 50, 100) | done |
| **C** | GRU ablation (N = 20, 50, 100) | done |
| **D** | Figures + summary table | done |

**Models:** XGBoost (tabular, Week 4 champion) and single-head GRU (Week 5 sequence model). Week 6 monotonic GRU is out of scope.

In [ ]:
import os

# Set before numpy/sklearn/xgboost/torch import OpenMP (Mac Jupyter deadlock fix)
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find data/raw/ starting from {here}')


ROOT = find_repo_root()
FEATURES_PATH = ROOT / 'data' / 'processed' / 'cell_features.csv'
SPLIT_PATH = ROOT / 'data' / 'processed' / 'cell_split.csv'
SUMMARY_PATH = ROOT / 'data' / 'processed' / 'cycle_summary.csv'
METRICS_DIR = ROOT / 'results' / 'metrics'
FIGURES_DIR = ROOT / 'results' / 'figures'

LABEL_COLS = ('file_id', 'cell_id', 'EOL', 'initial_capacity')
TARGET = 'EOL'
RANDOM_STATE = 42
WINDOW_CYCLES = (20, 50, 100)
XGB_MAX_DEPTHS = [3, 5, 7]
XGB_LEARNING_RATES = [0.05, 0.1]
CYCLE_MIN = 1  # exclude formation cycle 0
SEQUENCE_COLS = (
    'soh',
    'dc_internal_resistance',
    'energy_efficiency',
    'temperature_average',
)
N_CHANNELS = len(SEQUENCE_COLS)

print('Project root:', ROOT)
print('Windows N:', WINDOW_CYCLES)
print('Split:', SPLIT_PATH)

## Step A — Feature subsets per window

Rule: only use columns that could be computed from cycles **1…N** (no peeking beyond the window).

- **Window stats** (`_w20`, `_w50`, `_w100`): include suffix only if window end ≤ N.
- **Snapshots** (`capacity_c10`, `soh_c10`, …): include only if snapshot cycle ≤ N.
- **ΔV(Q):** `c10→c50` needs cycle 50; `c10→c100` needs cycle 100.

Expected counts: **12** features at N = 20, **28** at N = 50, **44** at N = 100 (Week 4 full matrix).

In [ ]:
cell_features = pd.read_csv(FEATURES_PATH)
feature_cols = [c for c in cell_features.columns if c not in LABEL_COLS]

assert len(cell_features) == 134
assert cell_features['file_id'].is_unique
assert cell_features[feature_cols].isna().sum().sum() == 0

print(f'Cells: {len(cell_features)}')
print(f'All features: {len(feature_cols)}')
print(f'EOL range: {cell_features[TARGET].min()} – {cell_features[TARGET].max()} cycles')

In [ ]:
def features_for_window(n: int, all_features: list[str]) -> list[str]:
    """Feature columns valid when only cycles 1..n are available."""
    if n not in WINDOW_CYCLES:
        raise ValueError(f'n must be one of {WINDOW_CYCLES}, got {n}')

    selected: list[str] = ['resistance_initial']

    for suffix, window_end in (('_w20', 20), ('_w50', 50), ('_w100', 100)):
        if window_end <= n:
            selected.extend([c for c in all_features if c.endswith(suffix)])

    for cycle in (10, 50, 100):
        if cycle <= n:
            selected.extend([f'capacity_c{cycle}', f'soh_c{cycle}'])

    if n >= 50:
        selected.extend(
            [c for c in all_features if c.startswith('delta_v_') and '_c10_c50' in c]
        )
    if n >= 100:
        selected.extend(
            [c for c in all_features if c.startswith('delta_v_') and '_c10_c100' in c]
        )

    selected_set = set(selected)
    return [c for c in all_features if c in selected_set]


FEATURES_BY_N = {n: features_for_window(n, feature_cols) for n in WINDOW_CYCLES}
EXPECTED_FEATURE_COUNTS = {20: 12, 50: 28, 100: 44}

for n in WINDOW_CYCLES:
    feats = FEATURES_BY_N[n]
    assert len(feats) == EXPECTED_FEATURE_COUNTS[n], (
        f'N={n}: expected {EXPECTED_FEATURE_COUNTS[n]} features, got {len(feats)}'
    )

assert FEATURES_BY_N[100] == feature_cols, 'N=100 must include all 44 features'

summary = pd.DataFrame(
    {
        'N': list(WINDOW_CYCLES),
        'n_features': [len(FEATURES_BY_N[n]) for n in WINDOW_CYCLES],
        'has_delta_v_c10_c50': [n >= 50 for n in WINDOW_CYCLES],
        'has_delta_v_c10_c100': [n >= 100 for n in WINDOW_CYCLES],
    }
)
print(summary.to_string(index=False))
print()
for n in WINDOW_CYCLES:
    print(f'N={n} ({len(FEATURES_BY_N[n])} features):')
    print(' ', ', '.join(FEATURES_BY_N[n]))
    print()

## Step A — Load split (do not re-split)

Reuse `cell_split.csv` from Week 4: **94 train / 20 val / 20 test** (`random_state=42`).

In [ ]:
if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        f'Missing {SPLIT_PATH}. Run notebooks/07_ml_baselines.ipynb first.'
    )

split_df = pd.read_csv(SPLIT_PATH)
assert set(split_df['split']) == {'train', 'val', 'test'}
assert len(split_df) == 134

data = cell_features.merge(split_df[['file_id', 'split']], on='file_id', validate='one_to_one')
assert len(data) == 134

train = data[data['split'] == 'train']
val = data[data['split'] == 'val']
test = data[data['split'] == 'test']

split_counts = split_df['split'].value_counts().sort_index()
print('Loaded split from', SPLIT_PATH)
print(split_counts.to_string())
print(f'train {len(train)} | val {len(val)} | test {len(test)}')

## Step A — GRU sequence windows (preview)

GRU inputs come from `cycle_summary.csv`: first **N** cycles × **4** channels (`SEQUENCE_COLS`). Steps B–C will build `(n_cells, N, 4)` tensors per window. Step A only checks that every cell has cycles 1…100 available.

In [ ]:
cycle_counts = (
    pd.read_csv(SUMMARY_PATH, usecols=['file_id', 'cycle_index'])
    .query('cycle_index >= @CYCLE_MIN')
    .groupby('file_id')['cycle_index']
    .max()
)

max_cycle_needed = max(WINDOW_CYCLES)
cells_with_enough_cycles = (cycle_counts >= max_cycle_needed).sum()

assert cells_with_enough_cycles == 134, (
    f'Expected 134 cells with cycle_index >= {max_cycle_needed}, '
    f'got {cells_with_enough_cycles}'
)

seq_preview = pd.DataFrame(
    {
        'N': list(WINDOW_CYCLES),
        'tensor_shape': [f'(134, {n}, {len(SEQUENCE_COLS)})' for n in WINDOW_CYCLES],
        'channels': [', '.join(SEQUENCE_COLS)] * len(WINDOW_CYCLES),
    }
)
print(f'All 134 cells have cycles {CYCLE_MIN}–{max_cycle_needed}')
print(seq_preview.to_string(index=False))

## Step A — Sanity summary

XGBoost at **N = 100** should reproduce Week 4 (about **85** test MAE). GRU at **N = 100** should reproduce Week 5 (about **111** test MAE). Steps B–C will verify after training.

In [ ]:
ablation_plan = pd.DataFrame(
    {
        'N': list(WINDOW_CYCLES),
        'xgboost_n_features': [len(FEATURES_BY_N[n]) for n in WINDOW_CYCLES],
        'gru_seq_len': list(WINDOW_CYCLES),
        'train_cells': [len(train)] * len(WINDOW_CYCLES),
        'val_cells': [len(val)] * len(WINDOW_CYCLES),
        'test_cells': [len(test)] * len(WINDOW_CYCLES),
    }
)
print('Week 7 ablation plan (Step A complete):')
print(ablation_plan.to_string(index=False))
print()
print('Step A complete — continue with Step B below.')

## Step B — XGBoost ablation

For each **N** ∈ {20, 50, 100}:

1. Subset features with `FEATURES_BY_N[N]`
2. Grid-search `max_depth` × `learning_rate` on **validation** MAE (same grid as Week 4)
3. Refit best model on train; report train / val / **test** metrics once
4. Save `results/metrics/xgboost_n{N}.json`

At **N = 100**, test MAE should be close to Week 4 (about **85** cycles).

In [ ]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def evaluate(model, X, y) -> dict[str, float]:
    return regression_metrics(y, model.predict(X))


def tune_xgboost(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
) -> dict:
    best_val_mae = np.inf
    best_params: dict = {}

    for max_depth in XGB_MAX_DEPTHS:
        for learning_rate in XGB_LEARNING_RATES:
            candidate = XGBRegressor(
                objective='reg:squarederror',
                n_estimators=300,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            candidate.fit(X_train, y_train)
            val_mae = evaluate(candidate, X_val, y_val)['mae']
            if val_mae < best_val_mae:
                best_val_mae = val_mae
                best_params = {
                    'max_depth': max_depth,
                    'learning_rate': learning_rate,
                }

    return best_params


def make_xgb_model(best_params: dict) -> XGBRegressor:
    return XGBRegressor(
        objective='reg:squarederror',
        n_estimators=300,
        max_depth=best_params['max_depth'],
        learning_rate=best_params['learning_rate'],
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

In [ ]:
METRICS_DIR.mkdir(parents=True, exist_ok=True)

xgb_results_by_n: dict[int, dict] = {}

for n in WINDOW_CYCLES:
    cols = FEATURES_BY_N[n]
    X_train_n = train[cols]
    y_train_n = train[TARGET]
    X_val_n = val[cols]
    y_val_n = val[TARGET]
    X_test_n = test[cols]
    y_test_n = test[TARGET]

    best_params = tune_xgboost(X_train_n, y_train_n, X_val_n, y_val_n)
    model = make_xgb_model(best_params)
    model.fit(X_train_n, y_train_n)

    metrics = {
        'model': 'xgboost',
        'window_cycles': n,
        'best_params': best_params,
        'n_features': len(cols),
        'features': cols,
        'split': {
            'train': len(train),
            'val': len(val),
            'test': len(test),
            'random_state': RANDOM_STATE,
        },
        'train': evaluate(model, X_train_n, y_train_n),
        'val': evaluate(model, X_val_n, y_val_n),
        'test': evaluate(model, X_test_n, y_test_n),
    }

    out_path = METRICS_DIR / f'xgboost_n{n}.json'
    out_path.write_text(json.dumps(metrics, indent=2))

    xgb_results_by_n[n] = metrics
    print(f'N={n}: best {best_params} | val MAE {metrics["val"]["mae"]:.1f} | test MAE {metrics["test"]["mae"]:.1f}')
    print(f'  saved {out_path}')

In [ ]:
xgb_summary = pd.DataFrame(
    {
        'N': list(WINDOW_CYCLES),
        'n_features': [xgb_results_by_n[n]['n_features'] for n in WINDOW_CYCLES],
        'val_MAE': [xgb_results_by_n[n]['val']['mae'] for n in WINDOW_CYCLES],
        'test_MAE': [xgb_results_by_n[n]['test']['mae'] for n in WINDOW_CYCLES],
        'test_RMSE': [xgb_results_by_n[n]['test']['rmse'] for n in WINDOW_CYCLES],
        'test_MAPE': [xgb_results_by_n[n]['test']['mape'] for n in WINDOW_CYCLES],
    }
).round(1)

print('XGBoost ablation — test set (20 cells):')
print(xgb_summary.to_string(index=False))
print()
print(f'N=100 test MAE: {xgb_results_by_n[100]["test"]["mae"]:.1f} (Week 4 reference: about 85)')
print('Step B complete — continue with Step C below.')

## Step C — GRU ablation

Single-head GRU from Week 5: first **N** cycles × **4** channels → predict EOL.

For each **N** ∈ {20, 50, 100}:

1. Build `(134, N, 4)` tensor from `cycle_summary.csv`
2. Grid-search 16 hyperparameter combos on **validation** MAE (same grid as Week 5)
3. Score train / val / **test** once; save `results/metrics/gru_sequence_n{N}.json`

At **N = 100**, test MAE should be close to Week 5 (about **111** cycles). Step C runs 48 training jobs (16 × 3 windows) — expect several minutes on CPU.

**Run via script** (recommended): `scripts/run_gru_ablation.py` — see **issue, diagnostics, and resolution** below for why. Smoke-test, then run the full grid in the cells below.

### Step C — issue, diagnostics, and resolution

**Original approach:** train the GRU grid **inside the notebook** (inline `train_gru_model` loop), same pattern as Week 5 notebook 08.

#### What went wrong

| Symptom | What it looked like |
|---------|---------------------|
| Stuck on epoch 1 | Heartbeat every 60s: `still training epoch 1/200 best val inf` — never finished epoch 1 |
| False “alive” signal | New log lines every minute, so it seemed to be working |
| No real work | Activity Monitor: Python **~0% CPU**, system mostly idle |
| Wrong speed | At **N=20**, one epoch should take **seconds** (6 batches × 94 cells), not 10+ minutes |

#### How we diagnosed it

1. **Stale Jupyter kernel** — Output still printed `still training … best val inf`, but that string had already been **removed** from the saved notebook. The kernel was running **old function definitions** from an earlier edit, not the code shown in the cells.
2. **Thread deadlock (Mac)** — A background heartbeat thread kept printing while the **main PyTorch thread** was blocked. Common cause: PyTorch + OpenMP + extra threads inside a long-lived Jupyter kernel.
3. **Control run outside Jupyter** — The same GRU logic in a fresh terminal Python process trained normally (~0.05 s/epoch at N=20). PyTorch and the data pipeline were fine; the **kernel environment** was the problem.
4. **Optional check in kernel** — `inspect.getsource(train_gru_model)` showed whether stale code (e.g. `threading`, `still training`) was still loaded.

#### Resolution (current design)

| Change | Why |
|--------|-----|
| **`scripts/run_gru_ablation.py`** | All GRU training runs in a **subprocess** via `run_gru_script()` — fresh Python each time, no stale defs |
| **`--smoke-test`** | One combo, one epoch (~5 s) before the full 3×16 grid |
| **Thread caps** | `OMP_NUM_THREADS=4` (and MKL/OpenBLAS) set in **cell 1** and the script **before** importing torch |
| **Quiet progress** | One line per grid combo (16 per window) — no per-epoch spam |

**If output looks wrong again:** Kernel → **Restart**, re-run from **cell 1**, then smoke test before the full run.

**Alternative:** from repo root, `python scripts/run_gru_ablation.py` (same as the notebook subprocess).

In [ ]:
import subprocess
import sys

GRU_SCRIPT = ROOT / 'scripts/run_gru_ablation.py'


def run_gru_script(*extra_args: str) -> None:
    """Run GRU ablation in a subprocess (same Python as this kernel)."""
    cmd = [sys.executable, str(GRU_SCRIPT), *extra_args]
    print('Running:', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(ROOT))
    if proc.returncode != 0:
        raise RuntimeError(f'{GRU_SCRIPT.name} failed with exit code {proc.returncode}')
    print('Script finished OK', flush=True)


print('GRU script:', GRU_SCRIPT)

In [ ]:
SUMMARY_COLS = [
    'file_id',
    'cell_id',
    'cycle_index',
    'discharge_capacity',
    'dc_internal_resistance',
    'energy_efficiency',
    'temperature_average',
]
cycle_summary_full = pd.read_csv(SUMMARY_PATH, usecols=SUMMARY_COLS)
cycle_summary_full = cycle_summary_full[cycle_summary_full['cycle_index'] >= CYCLE_MIN].copy()


def build_tensors_for_window(seq_len: int) -> np.ndarray:
    """Return X shape (134, seq_len, 4) in row order of `data`."""
    cycle_max = CYCLE_MIN + seq_len - 1
    sequences = []

    for row in data.itertuples(index=False):
        group = cycle_summary_full[cycle_summary_full['file_id'] == row.file_id]
        g = group[group['cycle_index'] <= cycle_max].sort_values('cycle_index')
        expected = np.arange(CYCLE_MIN, cycle_max + 1)
        if not np.array_equal(g['cycle_index'].to_numpy(), expected):
            missing = set(expected) - set(g['cycle_index'])
            raise ValueError(
                f'{row.file_id}: missing cycles {sorted(missing)[:5]}... (need {seq_len})'
            )

        soh = g['discharge_capacity'].to_numpy(dtype=float) / row.initial_capacity
        seq = np.column_stack([
            soh,
            g['dc_internal_resistance'].to_numpy(dtype=float),
            g['energy_efficiency'].to_numpy(dtype=float),
            g['temperature_average'].to_numpy(dtype=float),
        ])
        sequences.append(seq)

    X = np.stack(sequences, axis=0)
    assert X.shape == (len(data), seq_len, N_CHANNELS)
    assert not np.isnan(X).any()
    return X


y_all = data[TARGET].to_numpy(dtype=float)
split_arr = data['split'].to_numpy()

# Quick shape check
for n in WINDOW_CYCLES:
    X_n = build_tensors_for_window(n)
    print(f'N={n}: X shape {X_n.shape}')

### Step C — smoke test (~5 seconds)

Runs `scripts/run_gru_ablation.py --smoke-test` in a **subprocess**. You should see one combo line and `Smoke test OK`. Full run prints **16 lines per window** (one per hyperparameter combo), not per epoch.

In [ ]:
# Smoke test — must pass before full GRU run
run_gru_script('--smoke-test')

In [ ]:
# Step C — full GRU ablation (3 windows × 16 combos; long on CPU)
run_gru_script()

gru_results_by_n = {
    n: json.loads((METRICS_DIR / f'gru_sequence_n{n}.json').read_text())
    for n in WINDOW_CYCLES
}

for n in WINDOW_CYCLES:
    m = gru_results_by_n[n]
    print(
        f'N={n}: test MAE {m["test"]["mae"]:.1f} | '
        f'best {m["best_params"]}'
    )
print('Step C complete — continue with Step D (figures).')

In [ ]:
gru_summary = pd.DataFrame(
    {
        'N': list(WINDOW_CYCLES),
        'seq_len': list(WINDOW_CYCLES),
        'val_MAE': [gru_results_by_n[n]['val']['mae'] for n in WINDOW_CYCLES],
        'test_MAE': [gru_results_by_n[n]['test']['mae'] for n in WINDOW_CYCLES],
        'test_RMSE': [gru_results_by_n[n]['test']['rmse'] for n in WINDOW_CYCLES],
        'test_MAPE': [gru_results_by_n[n]['test']['mape'] for n in WINDOW_CYCLES],
    }
).round(1)

print('GRU ablation — test set (20 cells):')
print(gru_summary.to_string(index=False))
print()
print(f'N=100 test MAE: {gru_results_by_n[100]["test"]["mae"]:.1f} (Week 5 reference: about 111)')
print('Step C complete — continue with Step D (figures).')

## Step D — summary table and figure

Load saved metrics JSON from Steps B and C, build a combined **test-set** table, and save `results/figures/ablation_early_cycles.png` (grouped bar chart — test MAE vs window **N**).

In [ ]:
import matplotlib.pyplot as plt

ABLATION_FIGURE_PATH = ROOT / 'results' / 'figures' / 'ablation_early_cycles.png'


def load_ablation_metrics() -> tuple[dict[int, dict], dict[int, dict]]:
    xgb = {
        n: json.loads((METRICS_DIR / f'xgboost_n{n}.json').read_text())
        for n in WINDOW_CYCLES
    }
    gru = {
        n: json.loads((METRICS_DIR / f'gru_sequence_n{n}.json').read_text())
        for n in WINDOW_CYCLES
    }
    return xgb, gru


xgb_by_n, gru_by_n = load_ablation_metrics()

rows = []
for n in WINDOW_CYCLES:
    for label, blob in [('XGBoost', xgb_by_n[n]), ('GRU', gru_by_n[n])]:
        rows.append({
            'N': n,
            'model': label,
            'n_features_or_seq': blob.get('n_features', blob.get('sequence_len')),
            'val_MAE': blob['val']['mae'],
            'test_MAE': blob['test']['mae'],
            'test_RMSE': blob['test']['rmse'],
            'test_MAPE': blob['test']['mape'],
        })

ablation_summary = pd.DataFrame(rows).round(1)
print('Early-cycle ablation — test set (20 cells):')
print(ablation_summary.to_string(index=False))
print()
print('Week 4/5 reference at N=100: XGBoost ~85 MAE, GRU ~111 MAE')

In [ ]:
x = np.arange(len(WINDOW_CYCLES))
width = 0.35

xgb_mae = [xgb_by_n[n]['test']['mae'] for n in WINDOW_CYCLES]
gru_mae = [gru_by_n[n]['test']['mae'] for n in WINDOW_CYCLES]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width / 2, xgb_mae, width, label='XGBoost', color='C3')
ax.bar(x + width / 2, gru_mae, width, label='GRU', color='C0')
ax.set_xticks(x)
ax.set_xticklabels([f'N = {n}' for n in WINDOW_CYCLES])
ax.set_ylabel('Test MAE (cycles)')
ax.set_title('Early-cycle window ablation — test set (n = 20 cells)')
ax.legend()
fig.tight_layout()

ABLATION_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(ABLATION_FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', ABLATION_FIGURE_PATH)

**Step D complete.** Week 7 notebook done.

| Output | Path |
|--------|------|
| XGBoost metrics (per N) | `results/metrics/xgboost_n{20,50,100}.json` |
| GRU metrics (per N) | `results/metrics/gru_sequence_n{20,50,100}.json` |
| Ablation figure | `results/figures/ablation_early_cycles.png` |

Housekeeping: `docs/week07/README.md`, `docs/slides/week07_notes.md`, report §5.4 / §6.4 / §7.